# Figura 7 · IRPF real frente a IRPF con la tarifa actualizada con el IPC

Este cuaderno crea la Figura 7 del trabajo con **Plotly**. El resultado es un archivo `figura7.html` que se coloca junto a `index.html` y se incrusta en la web con un iframe.

**Qué muestra.** Año a año, de 2015 a 2024, el IRPF que paga un contribuyente tipo con la tarifa real y el que pagaría si la tarifa, los mínimos y los gastos se hubieran actualizado con el IPC desde 2016. La franja entre las dos líneas es **lo que se paga de más** por no actualizarla. Con unos botones se elige el contribuyente: salario mediano, salario medio o una familia con dos sueldos (uno medio y uno mediano) y dos hijos.

**Datos.** La tabla `data/processed/modelo_fiscal_resultados.csv`, que genera el cuaderno principal (paso 7) con el modelo fiscal del repositorio. No se crea ninguna tabla nueva.

**Supuestos** (los del modelo fiscal): sin otras rentas ni deducciones; escala autonómica igual a la estatal; la reducción por rendimientos del trabajo no se actualiza, porque sus cambios ya los decidió el legislador. Es una estimación, no una cifra oficial.

## 1. Preparación

In [1]:
import os, subprocess
import pandas as pd
import plotly.graph_objects as go

REPO = "https://github.com/sortaria24-max/08MBID_TrabajoPractico.git"
if not os.path.exists("data/processed"):
    if os.path.exists("../data/processed"):         # ejecutado desde la carpeta notebooks/
        os.chdir("..")
    else:                                            # Colab: descargamos el repositorio
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
        os.chdir("08MBID_TrabajoPractico")

AZUL, NARANJA, TINTA, TINTA2, GRIS = "#2A78D6", "#EB6834", "#14130F", "#55534C", "#D9D6CC"

def euros(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

def pct(x):
    return f"{x:.1f}".replace(".", ",") + " %"

## 2. Datos

Leemos los resultados del modelo fiscal y comprobamos que la diferencia es el IRPF real menos el actualizado. En 2015 y 2016 las dos cifras coinciden: 2016 es el año de partida de la actualización.

In [2]:
r = pd.read_csv("data/processed/modelo_fiscal_resultados.csv")
assert (r.irpf_real - r.irpf_tarifa_indexada_ipc - r.diferencia).abs().max() < 0.02   # cifras redondeadas al céntimo

PERFILES = {"mediana_individual": "Salario mediano",
            "media_individual": "Salario medio",
            "familia_media_mas_mediana": "Familia: dos sueldos y dos hijos"}
DESCRIPCION = {"mediana_individual": "Asalariado soltero con el salario mediano de cada año",
               "media_individual": "Asalariado soltero con el salario medio de cada año",
               "familia_media_mas_mediana": "Pareja con un salario medio y uno mediano y dos hijos"}

resumen = (r.groupby("perfil")
             .agg(diferencia_2024=("diferencia", "last"), acumulado=("diferencia", "sum"))
             .loc[list(PERFILES)].round(0))
resumen

,diferencia_2024,acumulado
perfil,,
mediana_individual,554.0,1805.0
media_individual,828.0,2710.0
familia_media_mas_mediana,1612.0,5306.0


## 3. Gráfico interactivo

- **Línea naranja:** el IRPF real de cada año.
- **Línea azul discontinua:** el IRPF con la tarifa actualizada con el IPC.
- **Franja naranja clara:** la diferencia, lo que se paga de más.
- **Recuadro de arriba:** la diferencia en 2024 y la suma de todos los años.
- **Botones:** cambian de contribuyente. Al pasar el ratón se ven las tres cifras de cada año.

In [3]:
fig = go.Figure()
capas, notas = {}, {}

for perfil, nombre in PERFILES.items():
    d = r[r.perfil == perfil].sort_values("anio")
    visible = perfil == "mediana_individual"
    ids = []
    fig.add_trace(go.Scatter(x=d.anio, y=d.irpf_tarifa_indexada_ipc, name="Con la tarifa actualizada",
        mode="lines+markers", line=dict(color=AZUL, width=2.5, dash="dash"), marker=dict(size=6),
        visible=visible, hovertemplate="Con la tarifa actualizada: <b>%{y:,.0f} €</b><extra></extra>"))
    ids.append(len(fig.data) - 1)
    texto = [f"Se paga de más: <b>{euros(v)}</b>" for v in d.diferencia]
    fig.add_trace(go.Scatter(x=d.anio, y=d.irpf_real, name="IRPF real", mode="lines+markers",
        line=dict(color=NARANJA, width=3), marker=dict(size=7), fill="tonexty",
        fillcolor="rgba(235,104,52,0.18)", visible=visible, text=texto,
        hovertemplate="IRPF real: <b>%{y:,.0f} €</b><br>%{text}<extra></extra>"))
    ids.append(len(fig.data) - 1)
    capas[perfil] = ids

    ult = d.iloc[-1]
    notas[perfil] = [
        dict(x=0, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom", yshift=14, align="left",
             showarrow=False, bgcolor="white", bordercolor=GRIS, borderwidth=1, borderpad=8,
             font=dict(size=13, color=TINTA),
             text=(f"<b>{nombre}</b> · {DESCRIPCION[perfil].lower()}<br>"
                   f"En 2024 paga <b>{euros(ult.diferencia)} más</b> de IRPF que con la tarifa actualizada "
                   f"({pct(ult.tipo_irpf_real_pct)} del bruto, en lugar de {pct(ult.tipo_irpf_indexado_pct)})<br>"
                   f"Sumando 2017-2024: <b>{euros(d.diferencia.sum())}</b> pagados de más")),
        dict(x=2024, y=ult.irpf_real, xanchor="left", xshift=10, showarrow=False, align="left",
             text=f"<b>IRPF real</b><br>{euros(ult.irpf_real)}", font=dict(size=12, color=NARANJA)),
        dict(x=2024, y=ult.irpf_tarifa_indexada_ipc, xanchor="left", xshift=10, showarrow=False, align="left",
             text=f"<b>Tarifa actualizada</b><br>{euros(ult.irpf_tarifa_indexada_ipc)}", font=dict(size=12, color=AZUL)),
    ]

botones = []
for perfil, nombre in PERFILES.items():
    vis = [False] * len(fig.data)
    for i in capas[perfil]:
        vis[i] = True
    botones.append(dict(label=nombre, method="update", args=[{"visible": vis}, {"annotations": notas[perfil]}]))

fig.update_layout(
    annotations=notas["mediana_individual"],
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=0,
                      x=0, y=1.30, xanchor="left", yanchor="bottom", pad=dict(r=6, t=0),
                      bgcolor="white", bordercolor=GRIS, font=dict(size=12, color=TINTA))],
    xaxis=dict(range=[2014.6, 2024.4], dtick=1, showgrid=False, zeroline=False, linecolor=TINTA2, fixedrange=True),
    yaxis=dict(title="IRPF al año", rangemode="tozero", tickformat=",.0f", ticksuffix=" €",
               gridcolor="#ECEAE3", zeroline=False, fixedrange=True),
    hovermode="x unified", hoverlabel=dict(font_size=13, bgcolor="white"), showlegend=False,
    font=dict(family="Inter, system-ui, -apple-system, Segoe UI, sans-serif", color=TINTA2, size=12),
    paper_bgcolor="white", plot_bgcolor="white", height=580, separators=",.",
    margin=dict(l=70, r=140, t=160, b=50),
)
fig.show(config={"displaylogo": False})

## 4. Guardar

- `figura7.html`: el gráfico interactivo, como página propia. Se coloca en StackBlitz junto a `index.html`, que lo incrusta con un iframe.
- Para el PDF: el icono de la cámara, arriba a la derecha del gráfico, descarga `figura7.png` con el contribuyente que se esté viendo (el salario mediano, por defecto). Súbelo a la carpeta `imagenes` del repositorio.

In [4]:
config = {"displaylogo": False, "responsive": True,
          "toImageButtonOptions": {"format": "png", "filename": "figura7", "scale": 2, "width": 1000, "height": 580}}
fig.write_html("figura7.html", include_plotlyjs="cdn", full_html=True, config=config)

# Misma tipografía que la web
html = open("figura7.html", encoding="utf-8").read()
fuente = ('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&display=swap">'
          '<style>body{margin:0}</style>')
open("figura7.html", "w", encoding="utf-8").write(html.replace("<head>", "<head>" + fuente, 1))
print("Guardado: figura7.html")

try:
    from google.colab import files
    files.download("figura7.html")
except ImportError:
    pass

Guardado: figura7.html
